# TIGER2D: 100 trials on the sampled 10,000-row dataset

Run `TIGER_sample_10000_and_create_6400_1600_2000_split.ipynb` first.

This notebook loads the exact saved:

- 6,400 training rows;
- 1,600 validation rows;
- 2,000 unseen rows.

It uses the official online TIGER context:

- 3 nt of 5′ context;
- 0 nt of 3′ context.

It also fixes the earlier `KeyError: '5p_context'` issue by accessing the
DataFrame columns directly rather than through `itertuples()` field names.

For each of 100 Optuna trials, it records only:

- validation MSE;
- unseen Pearson;
- unseen Spearman.

The best model is selected using validation MSE only.


In [ ]:
from pathlib import Path
import hashlib
import json
import random
import warnings

import numpy as np
import optuna
import pandas as pd
import tensorflow as tf

from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_squared_error

ACTIVITY_MODE = "observed_lfc"
# ACTIVITY_MODE = "negative_observed_lfc"

CONTEXT_5P = 3
CONTEXT_3P = 0

OPTUNA_SEED = 42
MODEL_SEED = 42
N_TRIALS = 100
MAX_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 10

BASE_OUTPUT_DIR = Path(
    f"results/tiger_repository_sampled_10000_{ACTIVITY_MODE}"
)
SPLIT_DIR = BASE_OUTPUT_DIR / "saved_splits"
RESULTS_DIR = BASE_OUTPUT_DIR / "trial_results_context_3_0"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = SPLIT_DIR / "train_split.csv"
VALIDATION_FILE = SPLIT_DIR / "validation_split.csv"
UNSEEN_FILE = SPLIT_DIR / "unseen_split.csv"
MANIFEST_FILE = BASE_OUTPUT_DIR / "split_manifest.json"

for path in [
    TRAIN_FILE,
    VALIDATION_FILE,
    UNSEEN_FILE,
    MANIFEST_FILE,
]:
    assert path.exists(), (
        f"Missing {path.resolve()}. "
        "Run the sampled split notebook first."
    )

print("TensorFlow:", tf.__version__)
print("Results:", RESULTS_DIR.resolve())


In [ ]:
def sha256(path):
    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


with open(MANIFEST_FILE, "r") as handle:
    manifest = json.load(handle)

if manifest["activity_mode"] != ACTIVITY_MODE:
    raise ValueError("ACTIVITY_MODE does not match the saved manifest.")

if (
    manifest["official_context_5p"] != CONTEXT_5P
    or manifest["official_context_3p"] != CONTEXT_3P
):
    raise ValueError(
        "Context settings do not match the saved manifest."
    )

assert sha256(TRAIN_FILE) == manifest["train_sha256"]
assert sha256(VALIDATION_FILE) == manifest["validation_sha256"]
assert sha256(UNSEEN_FILE) == manifest["unseen_sha256"]

train_data = pd.read_csv(TRAIN_FILE, keep_default_na=False)
validation_data = pd.read_csv(
    VALIDATION_FILE,
    keep_default_na=False,
)
unseen_data = pd.read_csv(
    UNSEEN_FILE,
    keep_default_na=False,
)

assert len(train_data) == 6400
assert len(validation_data) == 1600
assert len(unseen_data) == 2000

required_columns = {
    "guide_sequence",
    "target_sequence",
    "5p_context",
    "3p_context",
    "activity",
}

for subset_name, subset_df in [
    ("train", train_data),
    ("validation", validation_data),
    ("unseen", unseen_data),
]:
    missing_columns = required_columns - set(subset_df.columns)

    if missing_columns:
        raise ValueError(
            f"{subset_name} is missing: {sorted(missing_columns)}"
        )

    if not subset_df["5p_context"].str.len().eq(CONTEXT_5P).all():
        raise ValueError(
            f"{subset_name} does not contain exactly "
            f"{CONTEXT_5P}-nt 5′ contexts."
        )

    if not subset_df["3p_context"].str.len().eq(CONTEXT_3P).all():
        raise ValueError(
            f"{subset_name} does not contain exactly "
            f"{CONTEXT_3P}-nt 3′ contexts."
        )

MAX_GUIDE_LENGTH = int(
    max(
        train_data["guide_sequence"].str.len().max(),
        validation_data["guide_sequence"].str.len().max(),
        unseen_data["guide_sequence"].str.len().max(),
    )
)

TOTAL_LENGTH = CONTEXT_5P + MAX_GUIDE_LENGTH + CONTEXT_3P

print("Maximum guide/target length:", MAX_GUIDE_LENGTH)
print("Contextualized input length:", TOTAL_LENGTH)
print(
    "Subset sizes:",
    len(train_data),
    len(validation_data),
    len(unseen_data),
)


## Encode guide, target, and exact context

In [ ]:
BASE_TO_INDEX = {
    "A": 0,
    "C": 1,
    "G": 2,
    "T": 3,
}


def encode_base(
    output,
    sample_index,
    position,
    channel,
    base,
):
    if base in BASE_TO_INDEX:
        output[
            sample_index,
            BASE_TO_INDEX[base],
            position,
            channel,
        ] = 1.0


def encode_pairs_with_context(frame):
    encoded = np.zeros(
        (
            len(frame),
            4,
            TOTAL_LENGTH,
            2,
        ),
        dtype=np.float32,
    )

    guides = frame["guide_sequence"].astype(str).to_numpy()
    targets = frame["target_sequence"].astype(str).to_numpy()
    contexts_5p = frame["5p_context"].astype(str).to_numpy()
    contexts_3p = frame["3p_context"].astype(str).to_numpy()

    for row_index, (
        guide,
        target,
        context_5p,
        context_3p,
    ) in enumerate(
        zip(
            guides,
            targets,
            contexts_5p,
            contexts_3p,
        )
    ):
        guide = guide.strip().upper().replace("U", "T")
        target = target.strip().upper().replace("U", "T")
        context_5p = context_5p.strip().upper().replace("U", "T")
        context_3p = context_3p.strip().upper().replace("U", "T")

        if len(guide) != len(target):
            raise ValueError(
                f"Row {row_index}: guide and target lengths differ."
            )

        if len(context_5p) != CONTEXT_5P:
            raise ValueError(
                f"Row {row_index}: expected {CONTEXT_5P}-nt "
                f"5′ context, got {len(context_5p)}."
            )

        if len(context_3p) != CONTEXT_3P:
            raise ValueError(
                f"Row {row_index}: expected {CONTEXT_3P}-nt "
                f"3′ context, got {len(context_3p)}."
            )

        target_padded = target.ljust(MAX_GUIDE_LENGTH, "N")
        guide_padded = guide.ljust(MAX_GUIDE_LENGTH, "N")

        contextual_target = (
            context_5p
            + target_padded
            + context_3p
        )

        contextual_guide = (
            "N" * CONTEXT_5P
            + guide_padded
            + "N" * CONTEXT_3P
        )

        for position, base in enumerate(contextual_target):
            encode_base(
                encoded,
                row_index,
                position,
                0,
                base,
            )

        for position, base in enumerate(contextual_guide):
            encode_base(
                encoded,
                row_index,
                position,
                1,
                base,
            )

    return encoded


X_train = encode_pairs_with_context(train_data)
X_validation = encode_pairs_with_context(validation_data)
X_unseen = encode_pairs_with_context(unseen_data)

y_train = train_data["activity"].to_numpy(dtype=np.float32)
y_validation = validation_data["activity"].to_numpy(dtype=np.float32)
y_unseen = unseen_data["activity"].to_numpy(dtype=np.float32)

print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)
print("X_unseen:", X_unseen.shape)


In [ ]:
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


def safe_pearson(y_true, y_pred):
    if (
        len(y_true) < 2
        or np.std(y_true) == 0
        or np.std(y_pred) == 0
    ):
        return np.nan

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return float(pearsonr(y_true, y_pred)[0])


def safe_spearman(y_true, y_pred):
    if (
        len(y_true) < 2
        or np.std(y_true) == 0
        or np.std(y_pred) == 0
    ):
        return np.nan

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return float(spearmanr(y_true, y_pred)[0])


In [ ]:
def build_tiger2d(
    conv_filters,
    kernel_width,
    dense_1,
    dense_2,
    conv_dropout,
    dense_dropout_1,
    dense_dropout_2,
    learning_rate,
):
    inputs = tf.keras.Input(
        shape=(4, TOTAL_LENGTH, 2),
        name="contextualized_target_and_guide",
    )

    x = tf.keras.layers.Conv2D(
        filters=conv_filters,
        kernel_size=(4, kernel_width),
        activation="relu",
        padding="same",
    )(inputs)

    x = tf.keras.layers.Conv2D(
        filters=conv_filters,
        kernel_size=(4, kernel_width),
        activation="relu",
        padding="same",
    )(x)

    x = tf.keras.layers.MaxPool2D(
        pool_size=(1, 2),
        padding="same",
    )(x)

    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dropout(conv_dropout)(x)
    x = tf.keras.layers.Dense(dense_1, activation="sigmoid")(x)
    x = tf.keras.layers.Dropout(dense_dropout_1)(x)
    x = tf.keras.layers.Dense(dense_2, activation="sigmoid")(x)
    x = tf.keras.layers.Dropout(dense_dropout_2)(x)

    outputs = tf.keras.layers.Dense(
        1,
        activation="linear",
    )(x)

    model = tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="TIGER2D_sampled_10000",
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss=tf.keras.losses.LogCosh(),
        metrics=[
            tf.keras.metrics.MeanSquaredError(name="mse")
        ],
    )

    return model


## Run 100 fixed-split Optuna trials

In [ ]:
trial_records = []
trial_weights = {}


def objective(trial):
    tf.keras.backend.clear_session()
    set_all_seeds(MODEL_SEED)

    parameters = {
        "conv_filters": trial.suggest_categorical(
            "conv_filters",
            [16, 32, 48, 64],
        ),
        "kernel_width": trial.suggest_categorical(
            "kernel_width",
            [2, 3, 4, 5, 7],
        ),
        "dense_1": trial.suggest_categorical(
            "dense_1",
            [64, 96, 128, 160],
        ),
        "dense_2": trial.suggest_categorical(
            "dense_2",
            [16, 32, 48, 64],
        ),
        "conv_dropout": trial.suggest_float(
            "conv_dropout",
            0.10,
            0.50,
        ),
        "dense_dropout_1": trial.suggest_float(
            "dense_dropout_1",
            0.0,
            0.30,
        ),
        "dense_dropout_2": trial.suggest_float(
            "dense_dropout_2",
            0.0,
            0.30,
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate",
            1e-5,
            3e-3,
            log=True,
        ),
    }

    batch_size = trial.suggest_categorical(
        "batch_size",
        [32, 64, 128],
    )

    model = build_tiger2d(**parameters)

    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_mse",
        mode="min",
        patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
        verbose=0,
    )

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_validation, y_validation),
        epochs=MAX_EPOCHS,
        batch_size=batch_size,
        shuffle=True,
        callbacks=[early_stopping],
        verbose=0,
    )

    validation_predictions = model.predict(
        X_validation,
        batch_size=batch_size,
        verbose=0,
    ).reshape(-1)

    unseen_predictions = model.predict(
        X_unseen,
        batch_size=batch_size,
        verbose=0,
    ).reshape(-1)

    validation_mse = float(
        mean_squared_error(
            y_validation,
            validation_predictions,
        )
    )

    unseen_pearson = safe_pearson(
        y_unseen,
        unseen_predictions,
    )

    unseen_spearman = safe_spearman(
        y_unseen,
        unseen_predictions,
    )

    best_epoch = int(
        np.argmin(history.history["val_mse"]) + 1
    )

    trial.set_user_attr(
        "best_epoch",
        best_epoch,
    )

    trial_records.append(
        {
            "trial": trial.number,
            "validation_mse": validation_mse,
            "unseen_pearson": unseen_pearson,
            "unseen_spearman": unseen_spearman,
        }
    )

    trial_weights[trial.number] = [
        weight.copy()
        for weight in model.get_weights()
    ]

    print(
        f"Trial {trial.number:3d} | "
        f"Validation MSE: {validation_mse:.6f} | "
        f"Unseen Pearson: {unseen_pearson:.4f} | "
        f"Unseen Spearman: {unseen_spearman:.4f} | "
        f"Best epoch: {best_epoch}"
    )

    return validation_mse


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(
        seed=OPTUNA_SEED
    ),
    study_name=(
        "TIGER2D_sampled_10000_"
        "exact_context_100_trials"
    ),
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
)


## Save metrics and the best model

In [ ]:
results_df = (
    pd.DataFrame(trial_records)
    .sort_values("trial")
    .reset_index(drop=True)
)

results_df.to_csv(
    RESULTS_DIR / "all_100_trial_metrics.csv",
    index=False,
)

results_df["validation_mse"].to_csv(
    RESULTS_DIR / "Validation_loss.txt",
    index=False,
    header=False,
)

results_df["unseen_pearson"].to_csv(
    RESULTS_DIR / "Unseen_Pearson.txt",
    index=False,
    header=False,
)

results_df["unseen_spearman"].to_csv(
    RESULTS_DIR / "Unseen_Spearman.txt",
    index=False,
    header=False,
)

best_trial_number = study.best_trial.number
best_parameters = study.best_trial.params

best_model = build_tiger2d(
    conv_filters=best_parameters["conv_filters"],
    kernel_width=best_parameters["kernel_width"],
    dense_1=best_parameters["dense_1"],
    dense_2=best_parameters["dense_2"],
    conv_dropout=best_parameters["conv_dropout"],
    dense_dropout_1=best_parameters["dense_dropout_1"],
    dense_dropout_2=best_parameters["dense_dropout_2"],
    learning_rate=best_parameters["learning_rate"],
)

best_model.set_weights(
    trial_weights[best_trial_number]
)

best_model.save(
    RESULTS_DIR / "best_TIGER2D_sampled_10000_model.keras"
)

best_row = results_df.loc[
    results_df["trial"] == best_trial_number
].iloc[0]

with open(
    RESULTS_DIR / "best_trial_summary.json",
    "w",
) as handle:
    json.dump(
        {
            "activity_mode": ACTIVITY_MODE,
            "sampled_dataset_size": 10000,
            "n_train": 6400,
            "n_validation": 1600,
            "n_unseen": 2000,
            "context_5p": CONTEXT_5P,
            "context_3p": CONTEXT_3P,
            "best_trial": int(best_trial_number),
            "best_params": best_parameters,
            "best_epoch": int(
                study.best_trial.user_attrs["best_epoch"]
            ),
            "validation_mse": float(
                best_row["validation_mse"]
            ),
            "unseen_pearson": float(
                best_row["unseen_pearson"]
            ),
            "unseen_spearman": float(
                best_row["unseen_spearman"]
            ),
        },
        handle,
        indent=2,
    )

display(results_df.head())

print("Best trial:", best_trial_number)
print(best_row)
print("Saved to:", RESULTS_DIR.resolve())
